# 11 — Final YOLO11s · Kaggle submission

**Final training (separate from the three-fold CV).** YOLO11s pretrained on COCO, **6 epochs**, `imgsz=960`, `batch=2`, `AdamW lr0=0.0005`, `seed=42`. Use **all 56 official training sequences**, with **every third frame** as a practical subsample (~8k frames). The original seven `val/` sequences are **never used for training**. Their sampled images are used for validation, and the original `iscrowd` annotations are used in a separate COCOeval.

The resulting `submission.csv` covers the competition **test** images (not used for training or hyperparameter selection). `best.pt`, `last.pt`, metrics, and the submission are backed up as ZIP in `/kaggle/working` — **download the ZIP before closing Draft**. The previous 18-epoch public score 0.18731 is a different experiment, not directly comparable to COCO validation mAP.

**Run order:** setup → paths → training → holdout COCOeval → submission → backup. Only the training cell updates weights.

> **Status: unsuccessful run, not used in the final solution.** Downloading the pretrained `yolo11s.pt` from GitHub failed (HTTP 504), so training could not start; the run was then stopped manually. The final YOLO11s weights used later come from notebook 09 (18 epochs) and the CV folds of notebook 05. The two error tracebacks were removed from the outputs.

## 1. Packages and environment

In [1]:
%pip -q install ultralytics pycocotools pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from collections import defaultdict
import os, json, math, time, gc, random, shutil, zipfile
import numpy as np
import pandas as pd
import yaml
import torch
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED=42
CLASSES=['person','bicycle','car','van','truck','tricycle','bus','motor']
DATA_CANDIDATES=[
    Path('/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026'),
    Path('/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026'),
]
DATA=next((p for p in DATA_CANDIDATES if (p/'train/annotations_coco.json').exists()),None)
assert DATA is not None, 'Use Kaggle Add Input to attach the competition dataset.'
assert torch.cuda.is_available(), 'Kaggle Settings: enable GPU.'
ROOT=Path('/kaggle/working/final_yolo11s_6ep')
SPLITS=ROOT/'splits'; RUNS=ROOT/'runs'; REPORTS=ROOT/'reports'
for p in (SPLITS,RUNS,REPORTS): p.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Dataset:',DATA,'GPU:',torch.cuda.get_device_name(0))
print('Output:',ROOT)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 GPU: Tesla T4
Output: /kaggle/working/final_yolo11s_6ep


In [10]:
model = YOLO('yolo11s.pt')

WARNING ⚠️ Download failure, retrying 1/3 in 5s https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt
WARNING ⚠️ Download failure, retrying 2/3 in 10s https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt
WARNING ⚠️ Download failure, retrying 3/3 in 20s https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt


## 2. Official train/holdout split and verified image/label files

In [3]:
TRAIN_JSON = DATA/'train/annotations_coco.json'
VAL_JSON = DATA/'val/annotations_coco.json'
assert TRAIN_JSON.is_file() and VAL_JSON.is_file()
with TRAIN_JSON.open() as f: train_gt = json.load(f)
with VAL_JSON.open() as f: val_gt = json.load(f)

def as_table(d):
    rows = []
    for im in d['images']:
        fn = im['file_name']
        rows.append({'image_id': im['id'], 'file_name': fn,
                     'clip': Path(fn).parent.name, 'width': im['width'], 'height': im['height']})
    return pd.DataFrame(rows).sort_values(['clip','file_name']).reset_index(drop=True)

def sample_by_clip(d, step):
    return (d.groupby('clip', group_keys=False)
             .apply(lambda x: x.iloc[::step], include_groups=False)
             .reset_index(drop=True).assign(clip=lambda x: x['file_name'].map(lambda s: Path(s).parent.name)))

train_meta, holdout_meta = as_table(train_gt), as_table(val_gt)
TRAIN_STEP = 3          # All 56 train sequences, every third frame.
HOLDOUT_STEP = 3        # Seven entirely separate validation sequences.
train_selected = sample_by_clip(train_meta, TRAIN_STEP)
val_selected = sample_by_clip(holdout_meta, HOLDOUT_STEP)
assert set(train_meta['clip']).isdisjoint(set(holdout_meta['clip'])), 'Train / holdout video leakage!'
assert set(int(c['id']) for c in train_gt['categories']) == set(range(1,9))
assert set(int(c['id']) for c in val_gt['categories']) == set(range(1,9))
assert len(train_selected)>5000 and len(val_selected)>400
print('Train images:', len(train_meta), '=> sampled:',len(train_selected), '| clips:',train_meta['clip'].nunique())
print('Holdout images:',len(holdout_meta),' => sampled:',len(val_selected),'| clips:',holdout_meta['clip'].nunique())
print('Train clip overlap with holdout:',len(set(train_meta['clip']) & set(holdout_meta['clip'])))
train_selected.to_csv(SPLITS/'train_step3.csv',index=False)
val_selected.to_csv(SPLITS/'holdout_step3.csv',index=False)

Train images: 24201 => sampled: 8090 | clips: 56
Holdout images: 2846  => sampled: 951 | clips: 7
Train clip overlap with holdout: 0


In [4]:
def image_path(split, file_name):
    return DATA/split/file_name

def label_path(image):
    # Original format: split/images/<clip>/<frame>.jpg => split/labels/<clip>/<frame>.txt
    s = str(image)
    assert '/images/' in s, f'Unexpected image path: {s}'
    return Path(s.replace('/images/','/labels/')).with_suffix('.txt')

paths = {'train': [image_path('train',x) for x in train_selected['file_name']],
         'val':   [image_path('val',x) for x in val_selected['file_name']]}
for split, pp in paths.items():
    missing_img = [str(p) for p in pp if not p.is_file()]
    missing_lbl = [str(p) for p in pp if not label_path(p).is_file()]
    assert not missing_img, f'{split}: images missing: {missing_img[:3]}'
    assert not missing_lbl, f'{split}: labels missing: {missing_lbl[:3]}'
    (SPLITS/f'{split}.txt').write_text(chr(10).join(map(str, pp)) + chr(10))
    print(split, len(pp), 'images and labels verified')
DATA_YAML = SPLITS/'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'train':str(SPLITS/'train.txt'),
    'val':str(SPLITS/'val.txt'),
    'nc':8,
    'names':dict(enumerate(CLASSES)),
},sort_keys=False))
print('YAML:', DATA_YAML)

train 8090 images and labels verified
val 951 images and labels verified
YAML: /kaggle/working/final_yolo11s_6ep/splits/data.yaml


## 3. Training configuration and safe resume

These are the **selected 6-epoch CV hyperparameters**, adapted to the full set of sampled train sequences. Training starts from the official `yolo11s.pt` weights, not a fold-specific checkpoint. Unlike the previous longer 18-epoch run, this one is exactly 6 epochs. **Do not change settings in the middle of a run.**

The 6-epoch CV experiment used different augmentations from the later 18-epoch leaderboard attempt; this run fixes the settings explicitly for reproducibility.

In [5]:
RUN_NAME='yolo11s_960_step3_6ep_adamw_final'
CFG=dict(
    imgsz=960, epochs=6, batch=2, optimizer='AdamW', lr0=0.0005,
    lrf=0.1, weight_decay=0.0005, warmup_epochs=1.0, cos_lr=True,
    mosaic=0.0, mixup=0.0, degrees=0.0, translate=0.10,
    scale=0.30, fliplr=0.5, patience=6,
)
INIT_WEIGHTS='yolo11s.pt'
print('Run:',RUN_NAME,CFG)

Run: yolo11s_960_step3_6ep_adamw_final {'imgsz': 960, 'epochs': 6, 'batch': 2, 'optimizer': 'AdamW', 'lr0': 0.0005, 'lrf': 0.1, 'weight_decay': 0.0005, 'warmup_epochs': 1.0, 'cos_lr': True, 'mosaic': 0.0, 'mixup': 0.0, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.3, 'fliplr': 0.5, 'patience': 6}


In [6]:
def train_run(name, weights, cfg):
    run = RUNS/name
    best, last = run/'weights/best.pt', run/'weights/last.pt'
    history, marker = run/'results.csv',run/'completed.json'
    if marker.exists() and best.is_file():
        print('Already complete, skipping:',name)
        return best
    if history.exists() and best.is_file() and last.is_file() and len(pd.read_csv(history)) >= cfg['epochs']:
        marker.write_text(json.dumps({'name':name,'epochs':len(pd.read_csv(history)), 'best':str(best)},indent=2))
        print('Verified complete from history, skipping:',name)
        return best

    t0=time.monotonic()
    if last.is_file():
        print('Resuming interrupted run:',last)
        model=YOLO(str(last))
        model.train(resume=True,device=0,workers=2)
    else:
        print('Starting new training:',name)
        model=YOLO(str(weights))
        model.train(
            data=str(DATA_YAML), device=0, workers=2, cache=False,
            seed=SEED, deterministic=True, amp=True,
            save=True, save_period=-1, plots=False,
            project=str(RUNS),name=name,exist_ok=True,
            **cfg
        )
    assert best.exists(),f'Missing best.pt: {best}'
    assert history.exists(),f'Missing results.csv: {history}'
    n_epochs=len(pd.read_csv(history))
    assert n_epochs>=cfg['epochs'], f'Only {n_epochs}/{cfg["epochs"]} epochs; not marking complete.'
    marker.write_text(json.dumps({'name':name,'epochs':n_epochs,
                                  'init_weights':str(weights),'settings':cfg,
                                  'minutes_this_session':round((time.monotonic()-t0)/60,2)},indent=2))
    print('TRAINING COMPLETE:',name,'epochs:',n_epochs)
    del model
    gc.collect();torch.cuda.empty_cache()
    return best

### 3A — Train for 6 epochs (the only mandatory long GPU training cell)

If disconnected, rerun setup and this cell. It should resume from `last.pt` while the Draft files still exist. **Kaggle Draft output is temporary: download the backup ZIP later.**

In [7]:
best_ckpt=train_run(RUN_NAME, INIT_WEIGHTS, CFG)
assert best_ckpt.is_file()
assert (RUNS/RUN_NAME/'weights/last.pt').is_file()
print('Training completed:',best_ckpt)

Starting new training: yolo11s_960_step3_6ep_adamw_final
WARNING ⚠️ Download failure, retrying 1/3 in 5s https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt
WARNING ⚠️ Download failure, retrying 2/3 in 10s https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt
WARNING ⚠️ Download failure, retrying 3/3 in 20s https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt... 504 Server Error: Gateway Time-out for url: https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11s.pt


## 4. Independent COCOeval on seven original holdout videos (no training)

In [ ]:
HOLDOUT_COCO=COCO(str(VAL_JSON))
assert set(int(c['id']) for c in HOLDOUT_COCO.dataset['categories'])==set(range(1,9))

def evaluate_holdout(checkpoint, tag, imgsz=960, conf=0.001, iou=0.7, max_det=300, batch=4):
    checkpoint=Path(checkpoint)
    assert checkpoint.exists(),checkpoint
    folder=REPORTS/tag;folder.mkdir(parents=True,exist_ok=True)
    metrics_path=folder/'holdout_coco_metrics.json'
    preds_path=folder/'holdout_predictions_coco.json'
    if metrics_path.exists() and preds_path.exists():
        print('Evaluation cached:',metrics_path)
        return json.loads(metrics_path.read_text())
    model=YOLO(str(checkpoint))
    paths=[str(image_path('val',s)) for s in val_selected['file_name']]
    ids=[int(x) for x in val_selected['image_id']]
    preds=[]
    begin=time.monotonic()
    for i in range(0,len(paths),batch):
        rr=model.predict(source=paths[i:i+batch],imgsz=imgsz,device=0,
                         batch=batch,conf=conf,iou=iou,max_det=max_det,
                         augment=False,verbose=False,save=False)
        assert len(rr)==len(paths[i:i+batch])
        for im_id,r in zip(ids[i:i+batch],rr):
            if r.boxes is None: continue
            for box, cls, score in zip(r.boxes.xyxy.cpu().numpy(),
                                       r.boxes.cls.cpu().numpy(),
                                       r.boxes.conf.cpu().numpy()):
                x1,y1,x2,y2=map(float,box)
                w,h=max(0.,x2-x1),max(0.,y2-y1)
                if w<=0 or h<=0:continue
                assert 0<=int(cls)<8
                preds.append({'image_id':im_id,'category_id':int(cls)+1,
                              'bbox':[x1,y1,w,h],'score':float(score)})
        if (i//batch+1)%50==0:print('Evaluated images:',min(i+batch,len(paths)),'/',len(paths))
    assert len(preds)>0,'No detections. Verify checkpoint/confidence.'
    preds_path.write_text(json.dumps(preds))
    coco_dt=HOLDOUT_COCO.loadRes(preds)
    ev=COCOeval(HOLDOUT_COCO,coco_dt,'bbox')
    ev.params.imgIds=ids
    ev.params.catIds=list(range(1,9))
    ev.params.maxDets=[1,10,100]
    ev.evaluate();ev.accumulate();ev.summarize()
    ss=ev.stats
    result={
        'tag':tag,'checkpoint':str(checkpoint),
        'holdout_images':len(ids),'detections':len(preds),
        'mAP50-95':float(ss[0]),'mAP50':float(ss[1]),'mAP75':float(ss[2]),
        'AP_small':float(ss[3]),'AP_medium':float(ss[4]),'AP_large':float(ss[5]),
        'inference_ms_per_image':round((time.monotonic()-begin)*1000/len(ids),1),
        'imgsz':imgsz,'conf':conf,'iou':iou,'max_det_prediction':max_det,
        'coco_max_dets':100,'uses_original_iscrowd':True,
    }
    metrics_path.write_text(json.dumps(result,indent=2))
    print('Saved holdout metrics:',metrics_path)
    del model,ev,coco_dt
    gc.collect();torch.cuda.empty_cache()
    return result

In [ ]:
best_ckpt=RUNS/RUN_NAME/'weights/best.pt'
assert best_ckpt.is_file(), 'Train the model first.'
holdout_metrics=evaluate_holdout(best_ckpt,RUN_NAME,imgsz=CFG['imgsz'],conf=0.001,iou=0.7,max_det=300,batch=4)
display(pd.DataFrame([holdout_metrics]))

## 5. Generate a Kaggle submission (inference only)

PredictionString uses **0-based class IDs**, followed by `score x_min y_min width height`. This is the format already used in earlier accepted competition submissions. We verify test IDs, row count, and token shapes before saving.

In [ ]:
# Discover competition template if provided; otherwise use our established filename convention.
TEMPLATE_CANDIDATES=sorted(DATA.rglob('*sample*submission*.csv'))
print('Sample submission candidates:',TEMPLATE_CANDIDATES[:8])
TEST_IMAGES=sorted((DATA/'test/images').rglob('*.jpg')) + sorted((DATA/'test/images').rglob('*.png'))
TEST_IMAGES=sorted(set(TEST_IMAGES))
assert TEST_IMAGES, 'No test images found under test/images; inspect competition dataset layout.'

def test_image_id(path):
    # Previously produced CSV used e.g. clip_001_0000001
    return f'{path.parent.name}_{path.stem}'

if TEMPLATE_CANDIDATES:
    template=pd.read_csv(TEMPLATE_CANDIDATES[0])
    assert {'image_id','PredictionString'}.issubset(template.columns)
    wanted_ids=template['image_id'].astype(str).tolist()
else:
    template=None
    wanted_ids=[test_image_id(p) for p in TEST_IMAGES]

image_dict={test_image_id(p):p for p in TEST_IMAGES}
assert len(image_dict)==len(TEST_IMAGES), 'Duplicate test image IDs!'
assert len(wanted_ids)==len(set(wanted_ids)), 'Duplicate template IDs!'
assert set(wanted_ids)==set(image_dict), (
    f'Template/test mismatch: missing={list(set(wanted_ids)-set(image_dict))[:5]}, '
    f'extra={list(set(image_dict)-set(wanted_ids))[:5]}'
)
print('Verified test images:',len(TEST_IMAGES))
print('Sample IDs:',wanted_ids[:5])
print('Test IDs mapping verified.')

In [ ]:
SUBMIT_CHECKPOINT=RUNS/RUN_NAME/'weights/best.pt'
SUBMIT_IMGSZ=CFG['imgsz']
SUBMIT_CONF=0.01
SUBMIT_IOU=0.65
SUBMIT_MAX_DET=300
assert SUBMIT_CHECKPOINT.is_file()
print('Model for Kaggle submission:',SUBMIT_CHECKPOINT)

In [ ]:
SUBMISSION=ROOT/'submission.csv'
SUBMISSION_META=ROOT/'submission_settings.json'
if SUBMISSION.exists():
    raise RuntimeError('Existing submission.csv found. Keep it safe; rename it manually if generating a new submission.')

model=YOLO(str(SUBMIT_CHECKPOINT))
rows=[]
B=4
for start in range(0,len(wanted_ids),B):
    ids=wanted_ids[start:start+B]
    files=[str(image_dict[i]) for i in ids]
    results=model.predict(source=files, imgsz=SUBMIT_IMGSZ,device=0,batch=B,
                          conf=SUBMIT_CONF,iou=SUBMIT_IOU,max_det=SUBMIT_MAX_DET,
                          augment=False,verbose=False,save=False)
    assert len(results)==len(ids)
    for image_id,r in zip(ids,results):
        det=[]
        if r.boxes is not None:
            for box,cls,score in zip(r.boxes.xyxy.cpu().numpy(),r.boxes.cls.cpu().numpy(),r.boxes.conf.cpu().numpy()):
                x1,y1,x2,y2=map(float,box)
                w,h=max(0.,x2-x1),max(0.,y2-y1)
                if w<=0 or h<=0:continue
                cls=int(cls)
                assert 0<=cls<8
                det.extend([str(cls),f'{float(score):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}'])
        rows.append({'image_id':image_id,'PredictionString':' '.join(det)})
    if (start//B+1)%50==0:
        print('Test predictions:',min(start+B,len(wanted_ids)),'/',len(wanted_ids))

submission=pd.DataFrame(rows)
assert len(submission)==len(wanted_ids)
assert submission['image_id'].tolist()==wanted_ids
assert not submission['image_id'].duplicated().any()
assert submission['PredictionString'].notna().all()
# Check every detection is six tokens: cls score x y w h.
for s in submission['PredictionString']:
    t=s.split()
    assert len(t)%6==0
    if t:
        arr=np.array(t,dtype=float).reshape(-1,6)
        assert np.isfinite(arr).all()
        assert np.all((arr[:,0]>=0)&(arr[:,0]<8)&(np.floor(arr[:,0])==arr[:,0]))
        assert np.all((arr[:,1]>=0)&(arr[:,1]<=1))
        assert np.all(arr[:,4:6]>0)
submission.to_csv(SUBMISSION,index=False)
SUBMISSION_META.write_text(json.dumps({
    'checkpoint':str(SUBMIT_CHECKPOINT),'imgsz':SUBMIT_IMGSZ,
    'conf':SUBMIT_CONF,'iou':SUBMIT_IOU,'max_det':SUBMIT_MAX_DET,
    'n_test_images':len(wanted_ids),'format':'class confidence x_min y_min width height, YOLO 0-based classes'
},indent=2))
print('SUBMISSION WRITTEN:',SUBMISSION,'size MB',round(SUBMISSION.stat().st_size/1e6,1))
display(submission[['image_id']].head())
del model;gc.collect();torch.cuda.empty_cache()

## 6. Portable backup of checkpoints, metrics, YAML, and submission

**Run after training and inference.** This ZIP deliberately includes both `best.pt` and `last.pt` for future inference and training resume. Download the ZIP from Kaggle Output before ending the Draft session. Do not press Save & Run All solely to preserve a Draft's output.

In [ ]:
BACKUP=Path('/kaggle/working/final_yolo11s_6ep_all_essentials.zip')
RUN_DIR=RUNS/RUN_NAME
required=[RUN_DIR/'weights/best.pt', RUN_DIR/'weights/last.pt',
          RUN_DIR/'results.csv', DATA_YAML, ROOT/'submission.csv',
          REPORTS/RUN_NAME/'holdout_coco_metrics.json']
for p in required: assert p.is_file(), f'Backup prerequisite missing: {p}'
include=[]
for base in (RUN_DIR, REPORTS, SPLITS):
    for p in base.rglob('*'):
        if p.is_file() and (p.suffix in ('.pt','.csv','.json','.yaml','.txt')):
            # Avoid repetitive epoch snapshots; preserve best and last.
            if p.suffix == '.pt' and p.name not in ('best.pt','last.pt'): continue
            include.append(p)
for p in (ROOT/'submission.csv',ROOT/'submission_settings.json'):
    if p.is_file(): include.append(p)
include=sorted(set(include))
with zipfile.ZipFile(BACKUP,'w',compression=zipfile.ZIP_STORED) as z:
    for p in include:z.write(p,arcname=str(p.relative_to(ROOT)))
with zipfile.ZipFile(BACKUP) as z:
    assert z.testzip() is None
    names=set(z.namelist())
    assert any(s.endswith('/weights/best.pt') for s in names)
    assert any(s.endswith('/weights/last.pt') for s in names)
    assert 'submission.csv' in names
print('BACKUP READY:',BACKUP,'size MB:',round(BACKUP.stat().st_size/1e6,1),'files:',len(include))
print('IMPORTANT: download this ZIP outside the Kaggle Draft session.')

## 7. Results interpretation

- Compare the **original holdout** COCO metrics to other candidates evaluated on the same original holdout. CV metrics on different videos are not the same comparison.
- Do **not** train on test images. Use `submission.csv` only for Kaggle prediction upload.
- The Public Leaderboard score will only be known **after uploading `submission.csv`**. It may differ from holdout COCO mAP.
- 6 epochs do **not** automatically prevent overfitting. Check `results.csv` training/validation curves before deciding to extend training.
- Your previous 18-epoch YOLO score of 0.18731 remains a valid, separately trained comparator.